# Hálózatkészítés PCA-alapú hasonlósággal (Rich Months)

Ez a notebook a havi klímaadatokból készít hálózatot. A hónapok (csomópontok) közötti hasonlóságot egy PCA (főkomponens-analízis) alapú módszerrel határozzuk meg.

A folyamat lépései:
1. **Adatbetöltés**: Betöltjük a havi klímaadatokat (átlaghőmérsékletek, stb.).
2. **Adatmátrix előkészítése**: A havi adatokból egy mátrixot készítünk, ahol a sorok a hónapok, az oszlopok pedig a klimatikus változók.
3. **Standardizálás**: A változókat standardizáljuk (z-score transzformáció), hogy mindegyiknek 0 legyen az átlaga és 1 a szórása. Ez fontos a PCA számára.
4. **PCA**: Elvégezzük a főkomponens-analízist a standardizált adatokon.
5. **Score-ok kinyerése**: A PCA eredményéből kinyerjük az egyes hónapokhoz tartozó score-okat (koordinátákat a főkomponens-térben).
6. **Hasonlósági mátrix számítása**: A hónapok PC score-jai közötti euklideszi távolságot számoljuk. Ezt a távolságot egy [0, 1] intervallumra skálázzuk, majd hasonlósággá alakítjuk (1 - skálázott távolság).
7. **Hálózatépítés**: A számított hasonlósági mátrix alapján létrehozzuk a hálózatot, ahol egy küszöbértéknél nagyobb hasonlóság esetén él jön létre a hónapok között.

In [1]:
import sys
sys.path.append('../..')

import pandas as pd
import numpy as np
import networkx as nx
import itertools
import json
import os

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform

from networks.utils.get_rich_monthly_nodes import get_rich_monthly_nodes
from networks.utils.prune_to_average_degree import prune_to_average_degree

## 1. Hasonlósági Mátrix Számítása (PCA)

In [2]:
def calculate_pca_similarity_matrix(monthly_nodes, n_components=2):
    """
    Illeszt egy PCA modellt és kiszámítja a hasonlósági mátrixot.
    Visszaadja a mátrixot, a hónapokat, a súlyokat és az illesztett modelleket.
    
    Argumentumok:
    monthly_nodes (dict): Kulcsok a hónapok ('YYYY-MM'), értékek a havi klímaadatok.
    n_components (int): A PCA-hoz használt főkomponensek száma.
    
    Visszatérési érték:
    tuple: (similarity_matrix, months, component_weights, scaler, pca)
    """
    months = list(monthly_nodes.keys())
    sample_month_data = monthly_nodes[months[0]]
    feature_keys = [key for key, value in sample_month_data.items() if isinstance(value, (int, float))]
    
    print(f'Felhasznált változók a PCA-hoz: {feature_keys}')
    
    X = np.array([[monthly_nodes[month][key] for key in feature_keys] for month in months])
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X_scaled)
    
    print(f'Magyarázott variancia aránya a komponensekkel: {pca.explained_variance_ratio_}')
    
    # Komponens súlyok kinyerése
    component_weights = {}
    for i, component in enumerate(pca.components_):
        component_weights[f'component_{i+1}'] = dict(zip(feature_keys, component))
    
    distances = pdist(scores, metric='euclidean')
    
    if distances.max() - distances.min() > 0:
        scaled_distances = (distances - distances.min()) / (distances.max() - distances.min())
    else:
        scaled_distances = np.zeros_like(distances)
    
    similarity_matrix = 1 - squareform(scaled_distances)
    
    return similarity_matrix, months, component_weights, scaler, pca

def calculate_similarity_with_existing_pca(monthly_nodes, scaler, pca):
    """
    Kiszámítja a hasonlósági mátrixot egy előre illesztett PCA modell használatával.
    """
    months = list(monthly_nodes.keys())
    if not months:
        return np.array([]), []
        
    sample_month_data = monthly_nodes[months[0]]
    feature_keys = [key for key, value in sample_month_data.items() if isinstance(value, (int, float))]
    
    X = np.array([[monthly_nodes[month][key] for key in feature_keys] for month in months])
    
    X_scaled = scaler.transform(X)
    scores = pca.transform(X_scaled)
    
    distances = pdist(scores, metric='euclidean')
    
    if distances.max() - distances.min() > 0:
        scaled_distances = (distances - distances.min()) / (distances.max() - distances.min())
    else:
        scaled_distances = np.zeros_like(distances)
    
    similarity_matrix = 1 - squareform(scaled_distances)
    
    return similarity_matrix, months

## 2. Globális Hálózatok Létrehozása

In [3]:
def create_networks_with_shared_pca(cities, target_avg_degree_factor=3):
    threshold_json_filename = 'network_rich_thresholds.json'
    weights_json_filename = 'pca_component_weights.json'
    
    if os.path.exists(threshold_json_filename):
        with open(threshold_json_filename, 'r') as f: threshold_store = json.load(f)
    else: threshold_store = {}
        
    if os.path.exists(weights_json_filename):
        with open(weights_json_filename, 'r') as f: weights_store = json.load(f)
    else: weights_store = {}

    for city in cities:
        print(f'====== Feldolgozás: {city} ======')

        # 1. Hozzon létre egy közös PCA-teret a teljes időszak (1961-2024) adataiból
        print(f'--- Közös PCA-tér létrehozása a(z) {city} számára 1961-2024 adatokból ---')
        full_period_nodes = get_rich_monthly_nodes(city, '1961-01', '2024-12')
        
        if not full_period_nodes:
            print(f"FIGYELMEZTETÉS: Nincs adat a teljes időszakra a(z) {city} számára. A város kihagyása.")
            continue

        # Illesszen PCA-t a teljes adatsorra, hogy megkapja a közös tér modelleket és súlyokat
        full_sim_matrix, full_months, shared_weights, shared_scaler, shared_pca = calculate_pca_similarity_matrix(full_period_nodes)
        
        # Tárolja a közös súlyokat
        weights_key = f'{city}_shared_1961-2024'
        weights_store[weights_key] = shared_weights
        with open(weights_json_filename, 'w') as f: json.dump(weights_store, f, indent=4)
        print(f'PCA súlyok mentve a(z) {city} számára (közös): {weights_key}')

        # 2. Definiálja az időszakokat és dolgozza fel mindegyiket
        periods_to_process = [
            ('1961-01', '1990-12'), # korai
            ('1995-01', '2024-12'), # késői
            ('1961-01', '2024-12')  # teljes
        ]

        for start, end in periods_to_process:
            print(f'--- Hálózat építése: {city} : {start} - {end} ---')

            if start == '1961-01' and end == '2024-12':
                # Ez a teljes időszak, már rendelkezünk a mátrixszal
                similarity_matrix, months = full_sim_matrix, full_months
                period_nodes = full_period_nodes
            else:
                period_nodes = get_rich_monthly_nodes(city, start, end)
                if not period_nodes:
                    print(f"FIGYELMEZTETÉS: Nincs adat a(z) {start}-{end} időszakra a(z) {city} számára. Kihagyás.")
                    continue
                # Használja a közös PCA-teret a hasonlóság kiszámításához
                similarity_matrix, months = calculate_similarity_with_existing_pca(period_nodes, shared_scaler, shared_pca)

            # --- A többi a hálózatépítés, metszés és mentés ---
            json_key = f'{city}_{start}_{end}'
            
            G = nx.Graph()
            for month, data in period_nodes.items():
                G.add_node(month, **data)

            for i, j in itertools.combinations(range(len(months)), 2):
                score = similarity_matrix[i, j]
                if score > 0:
                    G.add_edge(months[i], months[j], weight=score)

            print(f'Kezdeti hálózat - Csomópontok: {G.number_of_nodes()}, Élek: {G.number_of_edges()}')

            num_years = int(end[:4]) - int(start[:4]) + 1
            target_avg_degree = num_years * target_avg_degree_factor - 1
            
            pruned_graph = prune_to_average_degree(G, target_avg_degree=target_avg_degree)

            if pruned_graph.number_of_edges() > 0:
                edge_weights = [d['weight'] for _, _, d in pruned_graph.edges(data=True)]
                final_threshold = float(min(edge_weights))
            else:
                final_threshold = 1.0
            
            threshold_store[json_key] = final_threshold
            with open(threshold_json_filename, 'w') as f: json.dump(threshold_store, f, indent=4)
            print(f'Küszöbérték ({final_threshold:.4f}) mentve: {json_key}')
            
            for node, data in pruned_graph.nodes(data=True):
                try:
                    year, month = str(node).split('-')
                    data['year'] = int(year)
                    data['month'] = int(month)
                except ValueError:
                    data['year'] = None
                    data['month'] = None
            
            output_dir = '../global_networks/rich_global/'
            os.makedirs(output_dir, exist_ok=True)
            pruned_path = os.path.join(output_dir, f'{city}_{start}_{end}.graphml')
            nx.write_graphml(pruned_graph, pruned_path)
            print(f'Hálózat mentve: {pruned_path}')
            print('-'*40)

## 3. Futtatás

In [4]:
cities_to_process = [
    'Cluj',
    'Bacskatopolya',
    'Brasov',
    'Deva',
    'Gheorgheni',
    'Gyor',
    'Kassa',
    'Kecskemet',
    'Keszthely',
    'Oradea',
    'Pecs'
]

# Futtatás
create_networks_with_shared_pca(cities_to_process)

====== Feldolgozás: Cluj ======
--- Közös PCA-tér létrehozása a(z) Cluj számára 1961-2024 adatokból ---
Felhasznált változók a PCA-hoz: ['mean_tn', 'mean_tx', 'mean_tg', 'rr_sum', 'mean_qq', 'mean_hu']
Magyarázott variancia aránya a komponensekkel: [0.77240082 0.14992698]
PCA súlyok mentve a(z) Cluj számára (közös): Cluj_shared_1961-2024
--- Hálózat építése: Cluj : 1961-01 - 1990-12 ---
Kezdeti hálózat - Csomópontok: 360, Élek: 64619
Pruning 48599 edges to reach average degree of 89...
Finished. Final Average Degree: 89.00
Küszöbérték (0.8155) mentve: Cluj_1961-01_1990-12
Hálózat mentve: ../global_networks/rich_global/Cluj_1961-01_1990-12.graphml
----------------------------------------
--- Hálózat építése: Cluj : 1995-01 - 2024-12 ---
Kezdeti hálózat - Csomópontok: 360, Élek: 64619
Pruning 48599 edges to reach average degree of 89...
Finished. Final Average Degree: 89.00
Küszöbérték (0.8175) mentve: Cluj_1995-01_2024-12
Hálózat mentve: ../global_networks/rich_global/Cluj_1995-01_2024-